<a href="https://colab.research.google.com/github/alex-jk/YRP-vehicle-accidents/blob/main/yrp_vehicle_accidents_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Import YRP data

In [18]:
from getpass import getpass
TOKEN = getpass('GitHub token: ')

!git clone https://{TOKEN}@github.com/alex-jk/YRP-vehicle-accidents.git
%cd YRP-vehicle-accidents

GitHub token: ··········
Cloning into 'YRP-vehicle-accidents'...
remote: Enumerating objects: 25, done.
remote: Counting objects: 100% (25/25), done.
remote: Compressing objects: 100% (23/23), done.
remote: Total 25 (delta 9), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (25/25), 10.75 KiB | 10.75 MiB/s, done.
Resolving deltas: 100% (9/9), done.
/content/YRP-vehicle-accidents/YRP-vehicle-accidents


In [19]:
import pandas as pd
import re
df = pd.read_csv("data/YRP Data vehicle accidents 2023 - now.csv")

# Parse year
df['Date'] = pd.to_datetime(df['Date'], errors='coerce')
df['Year'] = df['Date'].dt.year

# People per row: count items in Age/Gender (comma- or slash-separated); use max; default to 1
def count_items(x):
    if not isinstance(x, str): return 0
    parts = [p.strip() for p in re.split(r'[,/]', x) if p.strip()]
    return len(parts)

age_n    = df['Age'].apply(count_items)
gender_n = df['Gender'].apply(count_items)
person_n = age_n.combine(gender_n, max).where(lambda s: s>0, 1)
# print(person_n)

# Robust Y/N → boolean
is_yes = lambda x: isinstance(x, str) and x.strip().upper().startswith('Y')
ped = df['Pedestrian'].map(is_yes)
cyc = df['Cyclist'].map(is_yes)
mob = df['Mobility Scooter'].map(is_yes)

# Category per row (priority: Pedestrian > Cyclist > Mobility scooter > Other)
def victim_type(ped, cyc, mob):
    if ped: return 'Pedestrian'
    if cyc: return 'Cyclist'
    if mob: return 'Mobility Scooter'
    return 'Car'

df['Type'] = [victim_type(p, c, m) for p, c, m in zip(ped, cyc, mob)]
df['Deaths'] = person_n

print(f"\nDF shape: {df.shape}")
df.head(10)


DF shape: (68, 18)


,ID,Date,Time,Main Street,Main Street 2,Municipality,Intersection,Motorcycle,Age,Gender,Mobility Scooter,Pedestrian,Single Vehicle,Night,Cyclist,Year,Type,Deaths
0,2023_16704,2023-01-14,14:28,McCowan Road,Bur Oak Avenue,Markham,Y,N,75,Female,N,N,N,N,N,2023,Car,1
1,2023_35339,2023-01-30,6:30,9th Line,Bloomington Road,Whitchurch-Stouffville,Y,N,"Adult, Adult","Male, Male",N,N,N,N,N,2023,Car,2
2,2023_87652,2023-03-15,12:49,Highway 48,NaN,Whitchurch-Stouffville,N,N,35,Male,N,N,N,N,N,2023,Car,1
3,2023_170962,2023-05-19,23:49,Major MacKenzie Dr W,Jane Street,Vaughan,Y,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2023,Car,1
4,2023_175551,2023-05-23,18:11,Pine Valley Drive,Major MacKenzie Dr W,Vaughan,Y,N,20,Male,N,Y,N,N,N,2023,Pedestrian,1
5,2023_183025,2023-05-29,7:52,Jane Street,NaN,King,Y,Y,Adult,Male,N,N,N,N,N,2023,Car,1
6,2023_189685,2023-06-01,19:21,Pine Valley Drive,NaN,Vaughan,N,N,72,Female,N,N,N,N,N,2023,Car,1
7,2023_238781,2023-07-05,12:22,Dudley Avenue,NaN,Markham,N,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2023,Car,1
8,2023_265277,2023-07-25,22:15,Highway 7,Thornhill Woods Drive,Vaughan,Y,Y,"45, 47","Male, Female",N,N,N,N,N,2023,Car,2
9,2023_269592,2023-07-29,10:23,Keele Street,Sherwood Park Dr,Vaughan,Y,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2023,Car,1


Summarize by year and type

In [30]:
# Sum people by year & type
counts = (
    df.groupby(['Year','Type'])['Deaths']
      .sum()
      .unstack('Type')
      .fillna(0)
      .astype(int)
      .sort_index()
)

# totals (people) per year
year_totals = df.groupby('Year')['Deaths'].sum()

import re
import pandas as pd

# Add actual counts by gender (Male, Female, Unknown) per year

import re
import pandas as pd

def gender_counts(cell, deaths):
    # NA/blank → all Unknown
    if not isinstance(cell, str) or not cell.strip():
        return pd.Series({'Male Total': 0, 'Female Total': 0, 'Unknown Total': deaths})
    toks = [t.strip().lower() for t in re.split(r'[,/;|]', cell) if t.strip()]
    m = sum(t.startswith('m') for t in toks)        # M / Male
    f = sum(t.startswith('f') for t in toks)        # F / Female
    other = sum(not (t.startswith('m') or t.startswith('f')) for t in toks)
    u = other + max(0, deaths - (m + f + other))    # any extra people → Unknown
    return pd.Series({'Male Total': m, 'Female Total': f, 'Unknown Total': u})

# row → year totals
by_year_gender = (
    pd.concat([df['Year'],
               df.apply(lambda r: gender_counts(r['Gender'], int(r['Deaths'])), axis=1)], axis=1)
      .groupby('Year')[['Male Total','Female Total','Unknown Total']].sum()
)

# Make the table tidy, ordered, with totals
nice = counts.copy()

# attach to counts table
for col in ['Male Total','Female Total','Unknown Total']:
    nice[col] = by_year_gender.reindex(nice.index.drop('All years', errors='ignore')).get(col)

# "All years" totals
nice.loc['All years', ['Male Total','Female Total','Unknown Total']] = by_year_gender.sum().values

# ensure all expected columns exist
for col in ['Pedestrian', 'Cyclist', 'Mobility Scooter', 'Car', 'Male Total', 'Female Total', 'Unknown Total']:
    if col not in nice.columns:
        nice[col] = 0

# order columns and add totals
nice = nice[['Pedestrian', 'Cyclist', 'Mobility Scooter', 'Car', 'Male Total', 'Female Total', 'Unknown Total']]
nice['Total'] = nice[['Pedestrian', 'Cyclist', 'Mobility Scooter', 'Car']].sum(axis=1)
nice.loc['All years'] = nice.sum()

# ints + a simple style (optional in notebooks)
nice = nice.astype(int)
nice.style.set_caption("Fatalities by Year and Victim Type") \
          .format('{:,}')

Type,Pedestrian,Cyclist,Mobility Scooter,Car,Male Total,Female Total,Unknown Total,Total
Year,,,,,,,,
2023,5,0,0,18,11,7,5,23
2024,7,1,1,19,22,5,1,28
2025,5,1,0,13,11,4,4,19
All years,17,2,1,50,88,32,20,70
